# Carnatic Music Processing v2.0

### Imports

In [ ]:
import os
import librosa as lb
import librosa.display as ld
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import medfilt
from scipy.interpolate import interp1d
import scipy.stats as stats
from IPython.display import Audio


## Functions

### Read Dataset

In [ ]:
def load_dataset(directory_path, ftype = 'mp3'):
    '''
    Inputs:
    1. directory_path (STR)

    Outputs:
    1. data: (DICT) of form {id: TUPLE(file path (STR), audio (ARR), sampling rate (INT)))}
    '''
    data = {}
    try:
        items = [(directory_path + '/' + f) for f in os.listdir(directory_path) if f.endswith(ftype)]
    except FileNotFoundError:
        print(f"Error: Directory '{directory_path}' not found.")
        exit()
    for i in range(len(items)):
        item = items[i]
        x, sr = lb.load(item, sr = 44100)
        data[i] = (item, x, sr)
    return data

In [ ]:
# TEST
dataset = load_dataset('carnatic_varnam_1.0/Audio')

In [ ]:
def run_dataset(func, dataset):
    '''
    Given a function that generates output for a single audio file, loop over the entire dataset
    '''
    pass

### Spectrogram Generation

In [ ]:
kalyani1 = 'carnatic_varnam_1.0/Audio/223586__gopalkoduri__carnatic-varnam-by-badrinarayanan-in-kalyani-raaga.mp3'
k, ksr = lb.load(kalyani1, sr = 44100)

In [ ]:
Audio(data = k, rate = ksr)

In [ ]:
def clip_sample(x, sr, duration):
    '''
    Inputs:
    1. x: iterable object representing audio (ARR)
    2. sr: sampling rate at which x was taken (samples / sec) (INT)
    3. duration: desired duration (in seconds) of clipped output (FLOAT)

    Outputs:
    1. clip: iterable object representing clipped audio (ARR)
    '''
    if duration > 0 and duration <= (len(x) / sr): # check that clipping duration is valid
        samples = duration * sr # determine number of samples in clipped version
        clip = x[:samples]
    else:
        clip = x
    return clip

In [ ]:
def plot_spec(spec, sr, hop_length, n_fft = 2048, new = True):
    '''
    Inputs:
    1. spec: spectrogram (output of stft on audio) (2D ARR)
    2. sr: sampling rate (INT)
    3. hop_length: hop_length used when computing stft to generate spec (INT)
    4: n_fft: n_fft (window length?) used when computing stft to generate spec (INT)
    5: new: whether a new plot has to be initialized (BOOL)

    Outputs:
    1. display of spectrogram
    '''
    if new:
        plt.figure(figsize=(12, 6))
    # librosa has 'fft_svara' as an axis type :0
    lb.display.specshow(spec, sr=sr, hop_length = hop_length, n_fft = n_fft, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.show()

In [ ]:
def get_spec(x, sr, n_fft = 2048, hop_length = 512, duration = 25, display = True):
    '''
    Inputs:
    1. x: audio (ARR)
    2. sr: sampling rate (INT)
    3. hop_length: hop_length for stft to generate spec (INT)
    4: n_fft: n_fft (window length?) for stft to generate spec (INT)
    5: duration: length (in seconds) of audio being used (FLOAT)
    6: display: whether or not to output plot (BOOL)

    Outputs:
    1. spec (2D ARR)
    '''
    clip = clip_sample(x, sr, duration)
    spec = lb.amplitude_to_db(np.abs(lb.stft(clip, n_fft = n_fft, hop_length = hop_length)), ref=np.max)

    if display:
        plot_spec(spec, sr, hop_length, n_fft)
    return spec



In [ ]:
# TEST
kspec = get_spec(k, ksr, duration = 60)

### F0 Contour Generation
1. pYIN output (disconnected fragments)
2. smoothened and interpolated (for derivative calculations)

In [ ]:
def plot_f0(f0, sr, new = True):
    '''
    Inputs:
    1. f0: fundamental pitch contour (ARR)
    2. sr: sampling rate of audio for which f0 was generated (INT)
    5: new: whether a new plot has to be initialized (BOOL)

    Outputs:
    1. display of f0 contour
    '''
    if new:
        plt.figure(figsize=(12, 6))
    times = lb.times_like(f0, sr=sr)
    plt.plot(times, f0, label='f0 contour', color='r')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.title('Fundamental Frequency (f0) Contour')
    plt.legend()
    plt.show()

In [ ]:
def get_f0(x, sr, hop_length = 512, duration = 25, display = True):
    '''
    Inputs:
    1. x: audio (ARR)
    2. sr: sampling rate (INT)
    3. hop_length: hop_length for pyin (INT)
    4: duration: length (in seconds) of audio being used (FLOAT)
    5: display: whether or not to output plot (BOOL)

    Outputs:
    1. display of f0 contour
    '''
    clip = clip_sample(x, sr, duration)
    f0, voiced_flag, voiced_probs = lb.pyin(clip, fmin=lb.note_to_hz('C2'), fmax=lb.note_to_hz('C7'), hop_length=hop_length)

    if display:
        plot_f0(f0, sr)
    return f0

In [ ]:
# TEST
kf0 = get_f0(k, ksr, duration = 60)

In [ ]:
def med_filter(signal, kernel_size = 3):
    '''
    Inputs:
    1. signal: contour (ARR)
    2. kernel_size: size of neighborhood being used for smoothing (INT)

    Outputs:
    1. signal_filtered: smoothed (using median filtering) and filtered (using frequency bounds) version of input (ARR)
    '''
    # Apply median filtering to remove noise
    signal_filtered = medfilt(signal, kernel_size=kernel_size)

    # Remove out-of-bounds values
    fmin_hz = 75 # ranges from cepstrum paper
    fmax_hz = 750
    signal_filtered = np.where((signal_filtered > fmin_hz) & (signal_filtered < fmax_hz), signal_filtered, np.nan)
    return signal_filtered

def smooth_signal(signal, kernel_size=5):
    '''
    Inputs:
    1. signal: contour (ARR)
    2. kernel_size: size of neighborhood being used for smoothing (INT)

    Outputs:
    1. signal smoothed using averaging
    '''
    kernel = np.ones(kernel_size) / kernel_size
    return np.convolve(signal, kernel, mode='same')

In [ ]:
def smooth_f0(f0, sr, kernel_size = 3, hop_length = 512, display = True, new = True):
    '''
    Inputs:
    1. f0: input contour (ARR)
    2. sr: sampling rate of original audio (INT)
    3. kernel_size: size of neighborhood being used for smoothing (INT)
    4. hop_length: hop_length used to generate f0 from original audio with sampling rate sr (INT)
    5. display: (BOOL)
    6. new: (BOOL)

    Outputs:
    1. f0_smooth: smoothened f0 (ARR)
    '''
    # fmin and fmax roughly extrapolated from tonic ranges (110-250) to cover two octave range
    # Apply median filtering to remove noise
    f0_filtered = med_filter(f0, kernel_size)

    # Interpolate to fill missing values
    times = lb.times_like(f0, sr=sr, hop_length=hop_length)
    valid = ~np.isnan(f0_filtered)
    interp_func = interp1d(times[valid], f0_filtered[valid], kind='linear', fill_value="extrapolate")
    f0_smooth = interp_func(times)

    if display:
        plot_f0(f0_smooth, sr, new)
    return f0_smooth

In [ ]:
# TEST
kf0_smooth = smooth_f0(kf0, ksr)

In [ ]:
def plot_spec_f0(spec, f0, sr, hop_length = 512):
    '''
    Inputs:
    1. spec: (2D ARR)
    2. f0: (ARR)
    3. sr: sampling rate of original audio (INT)
    4. hop_length: hop_length used to generate f0 and spec (INT)

    Outputs:
    1. plot of spec with f0 overlaid
    '''

    times = lb.times_like(f0, sr=sr, hop_length=hop_length)

    # Plot spectrogram
    plt.figure(figsize=(12, 6))
    lb.display.specshow(spec, sr=sr, hop_length=512, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')

    # Overlay f0 contour
    plt.plot(times, f0, label='f0 contour', color='r', linewidth=2)

    # Labels and title
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.title('Spectrogram with f0 Contour')
    plt.legend()

    plt.show()

In [ ]:
# TEST
plot_spec_f0(kspec, kf0, ksr)
plot_spec_f0(kspec, kf0_smooth, ksr)

### Functions: Pitch Contour Features
1. Derivative
2. Zero Crossings (of the derivative)

In [ ]:
def get_derivative(f0, sr, hop_length = 512, smooth = True, kernel_size = 7):
    '''
    Inputs:
    1. f0: contour (ARR)
    2. sr: sampling rate of original audio (INT)
    3. hop_length: hop_length used to compute f0 (INT)
    4. smooth: whether to smooth the derivative contour using median filtering (BOOL)
    5. kernel_size: size of neighborhood being used for smoothing (INT)

    Outputs:
    1. f0_derivative: computed gradient, same length as f0 (ARR)
    2. f0_derivative_smooth: smoothened version of derivative if smooth is TRUE (else equivalent) (ARR)
    '''
        
    times = lb.times_like(f0, sr=sr, hop_length=hop_length)

    # Compute derivative of f0
    f0_derivative = np.gradient(f0, times) # Smooth the derivative to remove noise
    if smooth:
        f0_derivative_smooth = smooth_signal(f0_derivative, kernel_size=kernel_size)
    else:
        f0_derivative_smooth = f0_derivative
    return f0_derivative, f0_derivative_smooth

In [ ]:
def get_zero_crossings(f0_der, min_spacing = 5):
    '''
    Inputs:
    1. f0_der: derivative contour (ARR)
    2. min_spacing: minimum spacing between consecutive crossings to ignore noisy oscillations (INT)

    Outputs:
    1. zero_crossings: indices of f0_der at which a zero crossing crudely occurs (ARR)
    '''
    
    # Find sign changes
    sign_changes = np.diff(np.sign(f0_der))
    # Find indices where sign change is significant
    zero_crossings = np.where(np.abs(sign_changes) == 2)[0]  # Only strong sign flips
    # OPTIONAL: Further filter to ignore rapid oscillations (set minimum spacing)
    zero_crossings = zero_crossings[np.diff(np.concatenate(([0], zero_crossings))) > min_spacing]
    return zero_crossings

In [ ]:
def plot_der(f0, sr, f0_der, f0_der_smooth, zcs, hop_length = 512):
    '''
    Inputs:
    1. f0: pitch contour (ARR)
    2. sr: sampling rate of original audio (INT)
    3. f0_der: derivative of pitch contour (ARR)
    4. f0_der_smooth: smoothened derivative of pitch contour (ARR)
    5. zcs: zero crossings as a list of indices of f0_der? (ARR)
    2. hop_length: hop length used to compute all of these things from original audio (INT)

    Outputs:
    1. plot of f0 derivative and smoothened version with zero crossings marked as dots
    '''
        
    times = lb.times_like(f0, sr=sr, hop_length=hop_length)

    # Plot spectrogram with f0 contour
    plot_f0(f0, sr)

    plt.figure(figsize=(12, 8))
    # Plot derivative of f0
    plt.plot(times, f0_der, color='gray', alpha=0.5, label='Raw f0 Derivative')
    plt.plot(times, f0_der_smooth, color='g', label='Smoothed f0 Derivative')
    plt.axhline(0, color='k', linestyle='--', linewidth=1)  # Reference line at 0
    plt.scatter(times[zcs], f0_der_smooth[zcs], color='b', marker='o', label='Zero Derivative Points')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency Change (Hz/s)')
    plt.title('Derivative of f0 Contour')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# TEST
kf0_der, kf0_der_smooth = get_derivative(kf0, ksr)
kzcs = get_zero_crossings(kf0_der)
plot_der(kf0, ksr, kf0_der, kf0_der_smooth, kzcs)

### Function: Pitch Distribution Stats

In [ ]:
def plot_pitch_hist(pitch_midi, bins = 100, new = True):
    '''
    Inputs:
    1. pitch_midi: 
    2. bins: 
    3. new: 

    Outputs:
    1. plot of pitch histogram
    '''
    if new:
        plt.figure(figsize=(12, 6))
    plt.hist(pitch_midi, bins=bins, density=True, alpha=0.7, color='b', edgecolor='black')
    plt.xlabel('Frequency')
    plt.ylabel('Probability Density')
    plt.title('Pitch Histogram')    

def plot_kde(pitch_range, kde_values, new = True):
    if new:
        plt.figure(figsize=(12, 6))
    plt.plot(pitch_range, kde_values, color='r', linewidth=2)
    plt.xlabel('Frequency')
    plt.ylabel('Density')
    plt.title('Pitch Density (KDE)')

In [ ]:
def get_pitch_dist(f0, sr, hop_length = 512, filt_ker = 3, display = True):
    '''
    Inputs:
    1. f0: fundamental pitch contour (ARR)
    2. sr: sampling rate of original audio (INT)
    3. hop_length: hop_length used to compute f0 (INT)
    4. filt_ker: kernel size used for smoothing f0 (INT)
    5: display: (BOOL)

    Outputs:
    1. pitch_range: x-axis, linearly spaced points between min and max frequencies present (ARR)
    2. kde_values: probability density assigned to each x value in the pitch_range (ARR)
    3. if display: histogram of pitches with overlaid kde curve
    '''

    f0_clean = smooth_f0(f0, sr, filt_ker, hop_length, display = False)

    # Convert f0 to MIDI note numbers
    # pitch_midi = lb.hz_to_midi(f0_clean)

    # Compute Kernel Density Estimation (KDE)
    kde = stats.gaussian_kde(f0_clean)
    pitch_range = np.linspace(min(f0_clean), max(f0_clean), 500)
    kde_values = kde(pitch_range)

    if display:
        # Plot pitch histogram
        plt.figure(figsize=(12, 6))

        plot_pitch_hist(f0_clean, new = False)
        plot_kde(pitch_range, kde_values, new = False)
        plt.title('Pitch Distribution')
        plt.axvline(x=138.59/2, color='green', linestyle='-', label='S')
        plt.axvline(x=155.56/2, color='green', linestyle='-', label='R2')
        plt.axvline(x=174.61/2, color='green', linestyle='-', label='G3')
        plt.axvline(x=195.99/2, color='green', linestyle='-', label='M2')
        plt.axvline(x=207.4/2, color='green', linestyle='-', label='P')
        plt.axvline(x=233.08/2, color='green', linestyle='-', label='D2')
        plt.axvline(x=261.91/2, color='green', linestyle='-', label='N3')
        plt.axvline(x=138.59, color='green', linestyle=':', label='S')
        plt.axvline(x=155.56, color='green', linestyle=':', label='R2')
        plt.axvline(x=174.61, color='green', linestyle=':', label='G3')
        plt.axvline(x=195.99, color='green', linestyle=':', label='M2')
        plt.axvline(x=207.4, color='green', linestyle=':', label='P')
        plt.axvline(x=233.08, color='green', linestyle=':', label='D2')
        plt.axvline(x=261.91, color='green', linestyle=':', label='N3')
        plt.show()
    
    

    return pitch_range, kde_values

In [ ]:
# TEST
krange, kvalues = get_pitch_dist(kf0, ksr)

### Additional Functions: Note Transcription (2020)

In [ ]:
def compute_average_pitch_per_frame(f0, sr, hop_length = 512, frame_dur=0.025, overlap=0.5):
    '''
    Inputs:
    1. f0: fundamental pitch contour (ARR)
    2. sr: sampling rate of the original audio (INT)
    3. hop_length: hop length used to compute f0 (INT)
    4. frame_dur: length (in seconds) of each frame (FLOAT)
    5: overlap: proportion of overlap between frames (number 0-1) (FLOAT)

    Outputs:
    1. avg_f0_per_frame: average frequency per frame (ARR)
    2. fsr: effective sampling rate ofo the f0 contour = sr / hop_length (FLOAT)
    3. new_hop_length: hop_length used on f0 to generate averages (INT)
    '''

    # Effective sampling rate of the f0 contour being passed in
    fsr = sr/hop_length
    print(sr, hop_length, fsr)
    print(frame_dur*fsr)
    # Define frame length and hop length (50% overlap)
    frame_length = int(frame_dur * fsr)  # 25ms window
    print(frame_length)
    new_hop_length = max(int(frame_length * (1 - overlap)), 1) # 50% overlap

    # Compute average f0 per frame
    avg_f0_per_frame = []
    num_frames = int((len(f0) - frame_length) // new_hop_length) + 1

    for i in range(num_frames):
        start = i * new_hop_length
        end = start + frame_length

        # Extract current frame's f0 values (ignore NaNs)
        frame_f0 = f0[start:end]
        frame_f0 = frame_f0[~np.isnan(frame_f0)]  # Remove NaNs

        # Compute mean f0 if valid values exist, otherwise set NaN
        avg_f0_per_frame.append(np.mean(frame_f0) if len(frame_f0) > 0 else np.nan)

    avg_f0_per_frame = np.array(avg_f0_per_frame)
    # Compute times for each averaged frame
    return avg_f0_per_frame, fsr, new_hop_length

In [ ]:
def plot_segments(f0, sr, breakpoints, hop_length = 512):
    '''
    Inputs:
    1. f0: fundamental pitch contour (ARR)
    2. sr: sampling rate of the original audio (INT)
    3. breakpoints: segment boundaries as indices of f0 (ARR)
    4. hop_length: hop length used to compute f0 (INT)

    Outputs:
    1. plot of f0 with vertical boundaries at breakpoints
    '''
    
    # Compute time axis for f0 contour
    print(f0.shape)
    times = lb.times_like(f0, sr=sr, hop_length=hop_length)
    print('have times')
    # Plot f0 contour
    plt.figure(figsize=(12, 5))
    plt.plot(times, f0, label="f0 Contour", color='b', linewidth=2)

    # Add vertical lines for segment boundaries
    for idx in breakpoints:
        if 0 <= idx < len(times):  # Ensure the index is valid
            plt.axvline(times[idx], color='r', linestyle='-', linewidth=0.1, label="Segment Boundary" if idx == breakpoints[0] else "")

    # Labels and legend
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.title("Fundamental Frequency Contour with Segment Boundaries")
    plt.legend()
    plt.show()


In [ ]:
def segment_pitch_contour(f0, sr, f0_derivative, threshold, hop_length = 512, display = True):
    """
    Inputs:
    1. f0: fundamental pitch contour (ARR)
    2. sr: sampling rate of the original audio (INT)
    3. f0_derivative: derivative of f0 (ARR)
    4. threshold: magnitude threshold for derivative to cross for onset (FLOAT)
    5. hop_length: hop length used to compute f0 (INT)
    6. display: (BOOL)

    Outputs:
    1. breakpoints: indices at which the derivative crosses the threshold (ARR)
    2. segments: f0 segmented according to breakpoints (LIST[ARR])
    """
    # Find indices where the derivative exceeds the threshold
    breakpoints = np.where(np.abs(f0_derivative) > threshold)[0]
    print(f0.shape, breakpoints.shape)
    # Split f0 into segments
    segments = []
    start_idx = 0

    for bp in breakpoints:
        if bp > start_idx:  # Avoid empty segments
            segments.append(f0[start_idx:bp])
        start_idx = bp + 1  # Start next segment after the breakpoint

    # Append the last segment if it's not empty
    if start_idx < len(f0):
        segments.append(f0[start_idx:])

    if display:
        plot_segments(f0, sr, breakpoints, hop_length)
        
    return breakpoints, segments

In [ ]:
for l in range(5):
    kf0_avg, kfsr, kf0_avg_hop = compute_average_pitch_per_frame(kf0, ksr, frame_dur = 0.025*(l + 1)) # should we use some interpolated f0 to avoid nans? doesn't mess up averages within frames
    krange2, kvalues2 = get_pitch_dist(kf0_avg, kfsr, kf0_avg_hop)

In [ ]:
kf0_avg, kfsr, kf0_avg_hop = compute_average_pitch_per_frame(kf0, ksr) # should we use some interpolated f0 to avoid nans? doesn't mess up averages within frames

In [ ]:
# Check diff between using sr and fsr
krange2, kvalues2 = get_pitch_dist(kf0_avg, kfsr, kf0_avg_hop)

In [ ]:
kthresh = np.nanmean(np.abs(kf0_der_smooth))

In [ ]:
print(kf0_der_smooth)

In [ ]:
print(kthresh)

In [ ]:
kbps, ksegs = segment_pitch_contour(kf0, ksr, kf0_der, kthresh)

# Note Transcription (2020) Pipeline

Steps:
1. obtain audio recordings for ragas in the table (waiting on dataset from authors perhaps)
2. extract f0 using pyin
3. average pitch value for every frame (25 ms each, 50% overlap)
4. obtain PDF from values using Gaussian KDE
5. account for min/max values in bandwidth range (?)
6. smooth PDF with 5-point running average
7. identify tonic/base note (least variance & highest mean in 100-250 Hz range)
8. Plot derivative of pitch contour from phase I
9. Calculate onset derivative threshold (average derivative magnitude)
10. Segment by onsets
11. Label every segment

# Experiments

## 1. Statistics of Individual Notes
* use the Sonic Visualizer beat annotations and yaml notation to segment the audio into notes
* study properties of individual notes (mean, median, range, standard deviation, etc.)

### Parse Annotations

In [ ]:
import xml.etree.ElementTree as ET
import yaml

In [ ]:
def parse_svl(svl_file):
    '''
    Inputs:
    1. svl_file: (STR)

    Outputs:
    1. frames: frame indices of beat annotations (ARR)
    '''

    tree = ET.parse(svl_file)
    root = tree.getroot()

    frames = []
    for point in root.findall(".//point"):  # Iterate over all "point" elements
        frame = int(point.attrib["frame"])  # Extract the "time" attribute
        frames.append(frame)

    return frames

In [ ]:
def subdivide(arr, n, end=None):
    """
    Goal: Given an np array of integers, return a new array where each interval 
    is subdivided into n uniform regions, inserting (n-1) intermediate values.

    Inputs:
    1. arr: (ARR)
    2. n: number of subdivisions / multiplicative factor (INT)

    Outputs:
    1. result: arr with interspersed subdivisions (ARR)
    """
    if len(arr) < 2 or n < 1:
        return arr  # No subdivisions needed if there's 0 or 1 element or n < 1

    result = []
    for i in range(len(arr) - 1):
        start, end = arr[i], arr[i + 1]
        # Generate n points including start and end
        subdivided = np.linspace(start, end, n + 1, dtype=int)[:-1]  # Exclude the last point
        result.extend(subdivided)
    
    result.append(arr[-1])  # Append the last element explicitly
    if end != None:
        result.append(end)
    return np.array(result)

In [ ]:
ksvl = 'carnatic_varnam_1.0/Notations_Annotations/annotations/taalas/kalyani/badrinarayanan.svl'
kparsed_svl = parse_svl(ksvl)
kbeat_frames = subdivide(kparsed_svl, 2, len(k))

In [ ]:
def parse_yaml(file_path, mode=1):
    """
    Inputs:
    1. file_path: (STRING)
    2. mode: 1 for first half of varnam, 2 for entire thing (INT)

    Outputs:
    1. notes: list of string annotations for every akshara (LIST)
    """
    with open(file_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    def flatten(nested_list):
        """Recursively flattens a nested list structure."""
        flat_list = []
        for item in nested_list:
            if isinstance(item, list):
                flat_list.extend(flatten(item))
            else:
                flat_list.append(item)
        return flat_list

    # Extract and flatten required sections
    sections = ["pallavi", "anupallavi", "muktayiswaram"]
    notes = flatten([data.get(sec, []) for sec in sections])

    if mode == 1:
        return notes  # Return only pallavi, anupallavi, and muktayiswaram as a flat list

    elif mode == 2:
        notes.append("*")  # Add "*" after muktayiswaram
        notes = notes * 2
        notes.extend(["X"] * 64)

        # Process charanam and chittaiswaram in alternating fashion
        charanam = flatten(data.get("charanam", []))
        chittaiswarams = [flatten(swaram) for swaram in data.get("chittiswaram", [])]

        for swaram in chittaiswarams:
            notes.extend(charanam)  # Add charanam (without "*")
            notes.extend(swaram)    # Add chittaiswaram
            notes.append("*")       # Add "*" after chittaiswaram
            notes.extend(charanam)  # Add charanam (without "*")
            notes.extend(swaram)    # Add chittaiswaram
            notes.append("*") 
        notes.extend(["X"] * 64)

    return notes


In [ ]:
def subdivide_yaml(notes):
    """
    Goal: take yaml annotations and duplicate to account for second speed sections
    """
    doubled = True
    output = []
    for note in notes:
        if note == "*":
            doubled = not doubled
        else:
            output.append(note)
            if doubled:
                output.append(',')
    return output

In [ ]:
kyaml = 'carnatic_varnam_1.0/Notations_Annotations/notations/kalyani.yaml'
kparsed_yaml = parse_yaml(kyaml, 1)
knote_labels = kparsed_yaml

### Segment Audio

In [ ]:
def segment_audio(audio, timestamps, labels, f0, hop_length = 512):
    """
    Segment audio into chunks corresponding to each note using beat timestamps and pitch contour.

    Inputs:
    1. audio: input signal (ARR)
    2. timestamps: frame indices of audio corresponding to beat divisions (ARR)
    3. labels: list of note labels for each annotated beat, with ',' indicating sustained notes (LIST)
    4. f0: fundamental pitch contour (ARR)
    5. hop_length: hop length used to compute f0 (INT)

    Ouputs:
    1. note_segments: list of dictionaries representing each note as {"audio": (ARR), "label": (STR), "f0": (ARR)} (LIST[DICT])
    """
    note_segments = []
    start_idx = 0  # Index of first beat for each note
    curr_label = labels[0]
    
    last = False
    for i, label in enumerate(labels):
        if label != ',' and i > 0:
            frame1 = timestamps[start_idx]
            frame2 = timestamps[i]
            seg_audio = audio[frame1: frame2]
            f0frame1 = int(frame1 // hop_length)
            f0frame2 = int(frame2 // hop_length)
            seg_f0 = f0[f0frame1:f0frame2]
            note_segments.append({"audio": seg_audio, "label": curr_label, "f0": seg_f0, "start": frame1, "end": frame2})
            start_idx = i
            curr_label = label
        else:
            continue
            
    return note_segments


In [ ]:
kmidi = lb.hz_to_midi(kf0)
kmidi_smooth = lb.hz_to_midi(kf0_smooth)

In [ ]:
ksegments = segment_audio(k, kbeat_frames, knote_labels, kmidi_smooth)

In [ ]:
def get_swara_dict(segments, swaratype = {'S': 0, 'R': 2, 'G': 4, 'M': 5, 'P': 7, 'D': 9, 'N': 11}):
    '''
    Inputs:
        1. segments: list of dictionaries (one per segment) with audio, f0, and label
        2. swaratype: raga-dependent mapping of note label to position relative to tonic
    
    Outputs:
        1. swara_dict: maps semitone (0-11) to all f0 contours of segments with that label
    '''
    swara_dict = {k: [] for k in range(12)}
    for elt in segments:
        if elt['f0'].size != 0:
            swara_dict[swaratype[elt["label"][0]]].append(elt['f0'])
    return swara_dict
    


In [ ]:
kswara_type = {'S': 0, 'R': 2, 'G': 4, 'M': 6, 'P': 7, 'D': 9, 'N': 11}

In [ ]:
kswara_dict = get_swara_dict(ksegments, kswara_type)

### Plot Statistics

In [ ]:
def oct_inv(arr, tonic, center = 6):
    '''
    Inputs:
    1. arr: input array of MIDI values (ARR)
    2. tonic: tonic midi value (FLOAT)
    3. center: semitone value to center octave around (0-11 becomes -4 to 7 when centered around 2)

    Outputs:
    1. array with shifted/readjusted values
    '''
    shift = 6 - center
    return ((arr - tonic + shift) % 12) - shift

In [ ]:
# Write helper functions for plotting different types of charts

In [ ]:
def plot_stat(data, bins = 50, alpha = 0.5, chart = 'kde'):
    '''
    Plotting helper function, can do histograms or kde contours atm?

    Inputs:
    1. data: data lmao (ARR)
    2. bins: number of bins for histogram (INT)
    3. alpha: also a histogram parameter (opacity?) (FLOAT)
    4. chart: type of plot (STR)
    '''
    # Plot histograms for each key

    fmin = 1000
    fmax = 0
    for key, values in data.items():
        if values.size != 0:
            fmin = min(fmin, min(values))
            fmax = max(fmax, max(values))
    if chart == 'kde':
        x_values = np.linspace(fmin - 2, fmax + 2, 500)  # Define a range for x-axis
        for key, values in data.items():
            if values.size > 1: # kde needs more than one data point
                kde = stats.gaussian_kde(values)  # Compute KDE
                plt.plot(x_values, kde(x_values), label=key)  # Plot KDE

        # Adding labels and title
        plt.xlabel('Pitch')
        plt.ylabel('Density')
        plt.title('Gaussian KDEs for each key')
    
    elif chart == 'hist':
        for key, values in data.items():
            plt.hist(values, bins=10, alpha=0.5, label=key)

        # Adding labels and title
        plt.xlabel('Value')
        plt.ylabel('Pitch')
        plt.title('Histograms for each key')
    
    else:
        pass

    # Display legend
    plt.legend()

    # Show the plot
    plt.show()

In [ ]:
def get_stat(swara_dict, func, display = True, chart = 'kde', octinv = True, tonic = 69):
    '''
    Inputs:
    1. swara_dict: mapping of semitones to all f0 segments (preferrably in midi) of that note (DICT)
    2. func: function to be applied on each segment (FUNC)
    3. display: (BOOL)
    4. chart: (STR)
    5. octinv: whether or not to include octave invariance (BOOL)
    6. tonic: midi value of tonic (FLOAT)

    Outputs:
    1. stat_dict: mapping of semitones to list of outputs (func applied to each segment in the original list) (DICT)
    '''
    if octinv:
        stat_dict = {k: oct_inv(np.array([func(i) for i in v]), tonic, k) for k, v in swara_dict.items()}
    else:
        stat_dict = {k: np.array([func(i) for i in v]) for k, v in swara_dict.items()}
    if display:
        plot_stat(stat_dict, chart)
    return stat_dict

In [ ]:
# pretty straightforward statistics, renamed for ease of use

def stat_mean(arr):
    return np.mean(arr)

def stat_range(arr):
    return np.max(arr) - np.min(arr)

def stat_stdev(arr):
    return np.std(arr)

def stat_median(arr):
    return np.median(arr)

### Plots of Note Stats (Kalyani)

In [ ]:
# PLOT 1: MEAN PITCH
# no octave invariance, so densities of some notes are distributed across the whole octave
kmean_dict = get_stat(kswara_dict, stat_mean, octinv = False)

In [ ]:
# PLOT 2: MEAN PITCH
# using octave invariance and centering each note's contour around its central value
ktonic_hz = 138.59
ktonic_midi = lb.hz_to_midi(ktonic_hz)
kmean_dict_inv = get_stat(kswara_dict, stat_mean, tonic = ktonic_midi)


In [ ]:
# PLOT 3: STANDARD DEVIATION
# no dependence on mean frequency, all on same axes with no shift
# note S and P stdev is a bit lower (why is there such a wide spread for P?)

kstd_dict = get_stat(kswara_dict, stat_stdev)

In [ ]:
# PLOT 4: RANGE

krange_dict = get_stat(kswara_dict, stat_range)

In [ ]:
for seg in ksegments[:10]:
    print(seg["label"])
    display(Audio(data = seg["audio"], rate = ksr))

In [ ]:
for seg in ksegments:
    if stat_range(seg["f0"]) < 0:
        print(seg["label"], stat_range(seg["p0"]))
        display(Audio(data = seg["audio"], rate = ksr))

In [ ]:
i = 0
for seg in ksegments[:25]:
    print(i)
    if stat_stdev(seg["p0"]) > 10:
        print(seg["label"], stat_range(seg["p0"]))
        display(Audio(data = seg["audio"], rate = ksr))
    i += 1

### Pipeline for Multiple Audios

In [ ]:
dataset = load_dataset('carnatic_varnam_1.0/Audio')

In [ ]:
tonics = {'badrinarayanan': 138.59,
    'dharini': 200.58,
    'prasanna': 146.83,
    'ramakrishnamurthy': 149.4,
    'sreevidya': 210.07,
    'vignesh': 138.59}

ragadicts = {'abhogi': {'S': 0, 'R': 2, 'G': 3, 'M': 5, 'P': 7, 'D': 9, 'N': 10},
    'begada': {'S': 0, 'R': 2, 'G': 4, 'M': 5, 'P': 7, 'D': 9, 'N': 10},
    'kalyani': {'S': 0, 'R': 2, 'G': 4, 'M': 6, 'P': 7, 'D': 9, 'N': 11},
    'mohanam': {'S': 0, 'R': 2, 'G': 4, 'M': 5, 'P': 7, 'D': 9, 'N': 10},
    'sahana': {'S': 0, 'R': 2, 'G': 4, 'M': 5, 'P': 7, 'D': 9, 'N': 10},
    'saveri': {'S': 0, 'R': 1, 'G': 4, 'M': 5, 'P': 7, 'D': 8, 'N': 11},
    'sri': {'S': 0, 'R': 2, 'G': 3, 'M': 5, 'P': 7, 'D': 9, 'N': 10}}

In [ ]:
for k, v in dataset.items():
    # extract metadata from file name
    item, x, sr = v
    artist = item.split("-")[3]
    raga = item.split("-")[5]

    print(artist, raga)

In [ ]:
mean_dicts = []

for k, v in dataset.items():
    # extract metadata from file name
    item, x, sr = v
    artist = item.split("-")[3]
    raga = item.split("-")[5]

    print(artist, raga)
    # extract audio content and pitch contour
    xf0 = get_f0(x, sr, duration = 60, display = False)
    xf0_smooth = smooth_f0(xf0, sr)
    xmidi = lb.hz_to_midi(xf0)
    xmidi_smooth = lb.hz_to_midi(xf0_smooth)
    
    print('extracted audio')
    # get beat annotations from sonic visualizer
    xsvl = 'carnatic_varnam_1.0/Notations_Annotations/annotations/taalas/' + raga + '/' + artist + '.svl'
    xparsed_svl = parse_svl(xsvl)
    xbeat_frames = subdivide(xparsed_svl, 2, len(x))

    print('got beats')
    # get label notations from manual annotation
    xyaml = 'carnatic_varnam_1.0/Notations_Annotations/notations/' + raga + '.yaml'
    xparsed_yaml = parse_yaml(xyaml, 1)
    xnote_labels = xparsed_yaml

    xsegments = segment_audio(x, xbeat_frames, xnote_labels, xmidi_smooth)

    xswara_type = ragadicts[raga]
    xswara_dict = get_swara_dict(xsegments, xswara_type)

    xtonic_hz = tonics[artist]
    xtonic_midi = lb.hz_to_midi(xtonic_hz)
    for k, v in xswara_dict.items():
        print(k, len(v))
    xmean_dict_inv = get_stat(xswara_dict, stat_mean, tonic = xtonic_midi)
    mean_dicts.append((artist, raga, xmean_dict_inv))
    print('done! plot')


In [ ]:
print(len(mean_dicts))

In [ ]:
# merge same raga dicts?
# output mean and variance for each note, maybe nearest notes in both directions
all_means = []
all_stdevs = []
all_labels = []
all_ragas = []
for data in mean_dicts:
    sdict = data[2]
    for swara in sdict.keys():
        all_ragas.append(data[1])
        if len(sdict[swara] > 0):
            all_labels.append(swara)
            all_means.append(stat_mean(sdict[swara]))
            all_stdevs.append(stat_stdev(sdict[swara]))

In [ ]:
import seaborn as sns
sns.violinplot(x=all_labels, y=all_means, scale='width', cut=0)
plt.title('Violin Plot of Note Frequencies')
plt.show()

In [ ]:
unique_labels = np.unique(all_labels)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))

plt.figure(figsize=(4, 6))

for i, label in enumerate(unique_labels):
    data = np.array(all_means)[np.array(all_labels) == label]
    kde = stats.gaussian_kde(data)
    y_vals = np.linspace(min(data), max(data), 500)
    density = kde(y_vals)

    # Scale density to look nice and shift each slightly on x-axis if desired
    density = density / density.max()  # normalize to 1
    plt.fill_betweenx(y_vals, 0, density * 0.4, alpha=0.5, label=label, color=colors[i])

plt.xlabel('Density')
plt.ylabel('Frequency (Hz)')
plt.title('Overlapping KDEs by Note Label')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# why does looking at context matter? first plot statistics for merging all data known for a given semitone
